# Notebook 05 — Feature-Based Ensemble

Build a Poisson ensemble on top of the Dixon-Coles baseline (RPS 0.1646).
Strategy: engineer leakage-safe features, train gradient-boosted Poisson learners, stack with a penalized meta.

In [1]:
import sys
sys.path.append('..')   # so `src` is importable from notebooks/

import pandas as pd 
import numpy as np
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
import optuna
from scipy.stats import poisson
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import PoissonRegressor

# User-defined functions 
from src.evaluation import ranked_probability_score

## Load data

In [2]:
results = pd.read_parquet('../data/interim/results_clean.parquet')
print(results.head())
print(results.shape)

        date home_team away_team  home_score  away_score tournament     city  \
0 1872-11-30  Scotland   England           0           0   Friendly  Glasgow   
1 1873-03-08   England  Scotland           4           2   Friendly   London   
2 1874-03-07  Scotland   England           2           1   Friendly  Glasgow   
3 1875-03-06   England  Scotland           2           2   Friendly   London   
4 1876-03-04  Scotland   England           3           0   Friendly  Glasgow   

    country  neutral  
0  Scotland    False  
1   England    False  
2  Scotland    False  
3   England    False  
4  Scotland    False  
(49482, 9)


In [3]:
# Same data scope + split as Model 1, so both models are evaluated on the identical eval set
model_df = results[results['date'].dt.year >= 1970].reset_index(drop=True)

# Unique match key so per-team merges stay one-to-one (a team can play twice on the same date,
# which makes (date, team) non-unique and fans out the merges — match_id prevents that).
# insert at position 0 so it's the leading column.
model_df.insert(0, 'match_id', model_df.index)

## Feature 1 — As-of Elo rating

Run Elo chronologically and record each team's rating **before** the match update.
No leakage: the rating reflects only prior results. Tuned params from notebook 04: K=30, base=1500.

In [4]:
# As-of Elo feature: run Elo chronologically, record each team's rating BEFORE the match
def add_elo_features(matches, k=30, base_rating=1500):
    # Initialize ratings and elo lists
    ratings = {}
    elo_home_list = []
    elo_away_list = []

    for _, row in matches.iterrows():
        home_team = row['home_team']
        away_team = row['away_team']

        # Get current ratings or initialize them
        elo_home = ratings.get(home_team, base_rating)
        elo_away = ratings.get(away_team, base_rating)

        elo_home_list.append(elo_home)
        elo_away_list.append(elo_away)

        # Calculate expected 
        expected_home = 1 / (1 + 10 ** ((elo_away - elo_home) / 400))
        expected_away = 1 - expected_home

        # Determine actual result
        if row['home_score'] > row['away_score']:
            actual_home = 1
            actual_away = 0
        elif row['home_score'] == row['away_score']:
            actual_home = 0.5
            actual_away = 0.5
        else:
            actual_home = 0
            actual_away = 1

        # Add change
        change = k * (actual_home - expected_home)
        ratings[home_team] = elo_home + change
        ratings[away_team] = elo_away - change

    matches = matches.copy()
    matches['elo_home'] = elo_home_list
    matches['elo_away'] = elo_away_list
    
    return matches

model_df = add_elo_features(model_df)   # model_df must be chronologically sorted (it is)
print(model_df[['date', 'home_team', 'away_team', 'elo_home', 'elo_away']].head())

        date home_team       away_team  elo_home  elo_away
0 1970-01-04     Malta      Luxembourg    1500.0    1500.0
1 1970-01-14   England     Netherlands    1500.0    1500.0
2 1970-01-28    Israel     Netherlands    1500.0    1500.0
3 1970-02-04      Peru  Czechoslovakia    1500.0    1500.0
4 1970-02-06  Cameroon     Ivory Coast    1500.0    1500.0


## Feature 2 — Rolling form (goals scored / conceded)

Trailing 5-match rolling mean of goals scored and conceded per team.
Shifted by 1 (`shift(1).rolling(5).mean()`) so each row uses only prior matches — no leakage.
Computed on a long-format (one row per team per match) table, then merged back as home/away columns.

In [5]:
# Reshape to a per-team-per-match table for computing rolling form
def reshape_to_long_format(df):
    home_rows = df.rename(columns={
        'home_team': 'team',
        'home_score': 'goals_scored',
        'away_score': 'goals_conceded'
    })[['match_id', 'date', 'team', 'goals_scored', 'goals_conceded']]

    away_rows = df.rename(columns={
        'away_team': 'team',
        'away_score': 'goals_scored',
        'home_score': 'goals_conceded'
    })[['match_id', 'date', 'team', 'goals_scored', 'goals_conceded']]

    long_format = pd.concat([home_rows, away_rows], ignore_index=True)
    return long_format

long_df = reshape_to_long_format(model_df)
print(long_df.head())

   match_id       date      team  goals_scored  goals_conceded
0         0 1970-01-04     Malta             1               1
1         1 1970-01-14   England             0               0
2         2 1970-01-28    Israel             0               1
3         3 1970-02-04      Peru             0               2
4         4 1970-02-06  Cameroon             3               2


In [6]:
# Sort the long df by team and then by date
long_df = long_df.sort_values(by=['team', 'date'])

# Create trailing rolling mean
N = 5 
form = long_df.groupby('team')[['goals_scored', 'goals_conceded']].transform(lambda s: s.shift(1).rolling(N).mean())
long_df['form_scored'] = form['goals_scored']
long_df['form_conceded'] = form['goals_conceded']

# print(long_df) - Successful

# Merge form back to model_df 2 times, keyed on (match_id, team) so it's strictly one-to-one
# Home form 
home_form = long_df[['match_id', 'team', 'form_scored', 'form_conceded']].rename(columns={
    'team': 'home_team',
    'form_scored': 'home_form_scored',
    'form_conceded': 'home_form_conceded'
})

model_df = model_df.merge(home_form, on=['match_id', 'home_team'], how='left')

# Away form
away_form = long_df[['match_id', 'team', 'form_scored', 'form_conceded']].rename(columns={
    'team': 'away_team',
    'form_scored': 'away_form_scored',
    'form_conceded': 'away_form_conceded'
})

model_df = model_df.merge(away_form, on=['match_id', 'away_team'], how='left')

## Feature 3 — Rest days

Days since each team's previous match. Computed via `groupby('team')['date'].diff()` on the long table — naturally backward-looking, no leakage. NaN for a team's first appearance.

In [7]:
# Create rest days 
long_df['rest_days'] = long_df.groupby('team')['date'].diff().dt.days

# Merge back to model_df, keyed on (match_id, team) so it's strictly one-to-one
# Home rest days
home_rest_days = long_df[['match_id', 'team', 'rest_days']].rename(columns={
    'team': 'home_team',
    'rest_days': 'home_rest_days'
})

model_df = model_df.merge(home_rest_days, on=['match_id', 'home_team'], how='left')

# Away rest days
away_rest_days = long_df[['match_id', 'team', 'rest_days']].rename(columns={
    'team': 'away_team',
    'rest_days': 'away_rest_days'
})

model_df = model_df.merge(away_rest_days, on=['match_id', 'away_team'], how='left')

## Feature 4 — Tournament tier (ordinal)

Ordinal encoding of match importance: Friendly=0, Qualifier=1, Continental/regional cup=2, FIFA World Cup=3.
String matching on the `tournament` column — `"qualification"` catches all qualifiers, exact match for Friendly and FIFA World Cup, everything else falls to tier 2.

In [8]:
# Create a tournament tier 
def tournament_tier(name):
    if name == 'Friendly':
        return 0
    elif 'qualification' in name.lower():
        return 1
    elif name == 'FIFA World Cup':
        return 3
    else:
        return 2 # everything else = continental/regional Cup
    
model_df['tournament_tier'] = model_df['tournament'].apply(tournament_tier)

## Feature 5 — Head-to-head (matches played + goal difference)

Two features capturing the specific pairing's history, beyond what team-level Elo says:
- **`h2h_matches_played`** — count of prior meetings (the gate: tells the booster how much to trust the goal-diff).
- **`h2h_goal_diff`** — average goal difference across prior meetings, from the current home team's perspective. `NaN` if they've never met.

A match-up is symmetric but goal difference is directional, so we anchor on a **reference team** = the alphabetically-first of the pair (`key = tuple(sorted([home, away]))`). All history is stored in reference-team terms; we flip the sign twice — once on **read** (reference → home perspective) and once on **write** (home → reference perspective). Leakage-safe: features are appended *before* the current match updates the running dicts (same read→append→update rhythm as the Elo loop).

In [9]:
def add_h2h_features(matches):
    # Initialize count and goal difference
    h2h_matches_played = []
    h2h_goal_diff = []

    gd_sum = {} # cumulative goal diff from reference team's perspective
    count = {} # number of prior meetings

    for _, row in matches.iterrows():
        key = tuple(sorted([row['home_team'], row['away_team']]))
        reference_team = key[0]
        
        # Read the current state (before this match)
        prior_count = count.get(key, 0)
        prior_gd = gd_sum.get(key, 0) # reference team's perspective

        # Convert to home team's perspective and compute the feature
        if row['home_team'] == reference_team:
            home_perspective_gd = prior_gd
        else: 
            home_perspective_gd = -prior_gd

        if prior_count > 0:
            avg_gd = home_perspective_gd / prior_count
        else:
            avg_gd = np.nan

        # Append to lists
        h2h_matches_played.append(prior_count)
        h2h_goal_diff.append(avg_gd)

        # Update the dicts with this match's results (after the read, so no leakage)
        match_gd = row['home_score'] - row['away_score']
        if row['home_team'] != reference_team:
            match_gd = -match_gd
        
        gd_sum[key] = prior_gd + match_gd
        count[key] = prior_count + 1

    matches = matches.copy()
    matches['h2h_matches_played'] = h2h_matches_played
    matches['h2h_goal_diff'] = h2h_goal_diff
    return matches

model_df = add_h2h_features(model_df)

## Train / eval / WC2026 split

Placed **after** all feature engineering on purpose — every feature (Elo, rolling form, rest days, tournament tier, H2H) is computed on the full chronological `model_df` first, so cross-slice history is available. Splitting earlier would truncate a team's history and leak the split boundary into the feature values.

Same boundaries as notebooks 03 (Dixon-Coles) and 04 (Elo) so the ensemble is judged on the identical eval set:
- **train** — before 2024-01-01 (fit the boosters here)
- **eval** — 2024-01-01 → pre-WC-2026, WC excluded (tune + select models here)
- **wc2026** — the tournament itself, held aside; the sole test set

Boundaries are mutually exclusive so no match lands in two slices.

In [10]:
# Leakage-safe split — boundaries mutually exclusive so no match is in two slices
wc2026_mask = (model_df['tournament'] == 'FIFA World Cup') & (model_df['date'].dt.year == 2026)

train_df = model_df[model_df['date'] < '2024-01-01']                        # eval model sees only this
eval_df = model_df[(model_df['date'] >= '2024-01-01') & ~wc2026_mask]       # held-out test set (WC excluded)
wc2026_mask_df = model_df[wc2026_mask]                                      # forecast target, held aside

## Modeling approach — why almost no preprocessing

The base learners are **gradient-boosted trees** (XGBoost, LightGBM, CatBoost, HistGB), all with a Poisson objective. That choice removes most of the usual preprocessing pipeline:

- **No PolynomialFeatures / interaction terms.** Trees build interactions natively by splitting on one feature within a region defined by another. Explicit polynomial terms are pure redundancy.
- **No log transforms.** The Poisson objective already models `log(λ)` internally (log link), and trees are invariant to any monotonic transform of an input — they split on rank order, so `log(x)` and `x` give identical splits. Log-transforming inputs cannot change the fit.
- **No scaling / standardization.** Trees are scale-invariant; splits depend on order, not magnitude.
- **No imputation.** XGBoost, LightGBM, CatBoost and HistGB all handle `NaN` natively, learning a default split direction for missing values. The form/rest-days/H2H NaNs feed in untouched — the missingness is itself informative (a debut team, a never-met pairing).

So features go into the boosters **raw**.

### Feature selection — no RFE

We keep all engineered features and do **not** run RFE (recursive feature elimination):

1. The feature set is already small (~11) and hand-curated against the data ceiling — there is nothing to prune.
2. RFE is unstable with trees: correlated features (e.g. `elo_home`/`elo_away`, or Elo vs form, which both encode strength) split their importance arbitrarily, so RFE can drop a useful feature whose twin happened to absorb the credit that round. It's also expensive (N refits × CV).
3. Trees self-select — a useless feature simply never gets split on — and regularization handles the rest.

When we genuinely want to know whether a feature earns its keep (e.g. `tournament_tier`, `h2h_*`), we run a **targeted ablation**: drop it, refit, re-score **eval RPS**. Asking the metric we care about is more honest and decision-relevant than trusting a noisy importance score. Post-fit importances are used only as a **sanity diagnostic** (does Elo/form dominate?), never as a pruning trigger.

## Reshape to attacker/defender long format

Each match becomes **two rows** — one per scoring side — so a single Poisson model learns attack *and* defense at once, and the encoding is symmetric by construction (every team is seen as both attacker and defender). This is the ensemble analogue of the Dixon-Coles `reshape_to_long`, but carrying the engineered strength/form features instead of team dummies.

- **home-attacker row**: home team's features → `attacker_*`, away team's → `defender_*`, target `goals = home_score`, `is_home = 1` unless neutral.
- **away-attacker row**: the mirror — away team's features → `attacker_*`, target `goals = away_score`, `is_home = 0` (the away side never gets home advantage).

`tournament_tier`, `neutral`, and `h2h_matches_played` are **match-level** — identical in both rows. `h2h_goal_diff` is the one directional feature: it was stored in the home team's perspective, so it's **sign-flipped** on the away-attacker row. Doubles the training rows (38,900 → 77,800) — real leverage against the data ceiling.

In [11]:
# Create a function to reshape to long format for training
def reshape_to_long_training(df):
    # 1st subframe, home_attacker_rows
    home_attacker_rows = df.rename(columns={
        'elo_home': 'attacker_elo',
        'elo_away': 'defender_elo',
        'home_form_scored': 'attacker_form_scored',
        'home_form_conceded': 'attacker_form_conceded',
        'away_form_scored': 'defender_form_scored',
        'away_form_conceded': 'defender_form_conceded',
        'home_rest_days': 'attacker_rest_days',
        'away_rest_days': 'defender_rest_days',
        'home_score': 'goals'
    })[['match_id', 'date', 'attacker_elo', 'defender_elo', 'attacker_form_scored', 'attacker_form_conceded',
        'defender_form_scored', 'defender_form_conceded', 'attacker_rest_days', 'defender_rest_days',
        'tournament_tier', 'neutral', 'h2h_matches_played', 'h2h_goal_diff', 'goals']].copy()
    
    home_attacker_rows['is_home'] = (~home_attacker_rows['neutral']).astype(int)

    # 2nd subframe, away_attacker_rows — away team is the attacker, so every home/away pair swaps
    away_attacker_rows = df.rename(columns={
        'elo_away': 'attacker_elo',
        'elo_home': 'defender_elo',
        'away_form_scored': 'attacker_form_scored',
        'away_form_conceded': 'attacker_form_conceded',
        'home_form_scored': 'defender_form_scored',
        'home_form_conceded': 'defender_form_conceded',
        'away_rest_days': 'attacker_rest_days',
        'home_rest_days': 'defender_rest_days',
        'away_score': 'goals'
    })[['match_id', 'date', 'attacker_elo', 'defender_elo', 'attacker_form_scored', 'attacker_form_conceded',
        'defender_form_scored', 'defender_form_conceded', 'attacker_rest_days', 'defender_rest_days',
        'tournament_tier', 'neutral', 'h2h_matches_played', 'h2h_goal_diff', 'goals']].copy()
    
    away_attacker_rows['is_home'] = 0
    # h2h_goal_diff was stored in the home team's perspective; flip sign now that the attacker is the away team
    away_attacker_rows['h2h_goal_diff'] = -away_attacker_rows['h2h_goal_diff']

    return pd.concat([home_attacker_rows, away_attacker_rows], ignore_index=True)

long_train = reshape_to_long_training(train_df)

In [12]:
features = [
    'attacker_elo', 'defender_elo',
    'attacker_form_scored', 'attacker_form_conceded',
    'defender_form_scored', 'defender_form_conceded',
    'attacker_rest_days', 'defender_rest_days',
    'tournament_tier', 'neutral',
    'h2h_matches_played', 'h2h_goal_diff', 'is_home'
]

X_train = long_train[features]
y_train = long_train['goals']

## Evaluation — λ → scoreline → W/D/L → RPS

Same pipeline as Dixon-Coles (notebook 03), so every model is judged on the identical metric.

1. **Reshape** `eval_df` into the attacker/defender long format (each match → 2 rows).
2. **Predict λ** for every row. First half of the prediction vector = home-side λ's, second half = away-side λ's (the reshape concatenates in that order).
3. **Poisson PMF** on `[0, max_goals]` for each side → outer product → 11×11 scoreline probability matrix per match.
4. **W/D/L** from the matrix: lower triangle = home win, diagonal = draw, upper triangle = away win.
5. **RPS** against the one-hot actual outcome → mean across matches.

The booster's job is only to predict λ well; everything downstream is deterministic. Same evaluator works for any Poisson regressor (booster or GLM Pipeline) because it just calls `.predict()`.

In [13]:
# Create evaluate_booster function
def evaluate_booster(model, df, max_goals=10):
    # Reshape eval_df to long format
    long_eval = reshape_to_long_training(df)

    # Predict with model
    preds = model.predict(long_eval[features])

    n = len(df)
    lambda_home = preds[:n]
    lambda_away = preds[n:]

    home_score = df['home_score'].to_numpy()
    away_score = df['away_score'].to_numpy()

    rps_values = []
    for i in range(n):
        # Get home and away probs
        home_probs = poisson.pmf(np.arange(max_goals + 1), lambda_home[i])
        away_probs = poisson.pmf(np.arange(max_goals + 1), lambda_away[i])

        # Create matrix 
        matrix = np.outer(home_probs, away_probs)
        probs = [
            np.tril(matrix, -1).sum(),
            np.trace(matrix),
            np.triu(matrix, 1).sum()
        ]

        # One-Hot actual outcomes
        if home_score[i] > away_score[i]:
            actual = [1, 0, 0]
        elif home_score[i] == away_score[i]:
            actual = [0, 1, 0]
        else:
            actual = [0, 0, 1]

        value = ranked_probability_score(probs, actual)
        rps_values.append(value)

    # Mean rps
    mean_rps = np.mean(rps_values)
    return mean_rps

## Base learners — five Poisson regressors on defaults

Fit each with its native Poisson objective and **no tuning yet**. The goal here is a fair baseline snapshot:

- Default-vs-default is the honest way to compare learners — tuning one but not another confounds "better model" with "better tuning".
- Every one of them just needs to beat naive (0.227) to be worth keeping; anything close to Elo (0.1695) is a strong signal the feature set is doing real work; the target to chase after tuning is DC (0.1646).
- Trees eat raw features (no scaling, no imputation, no encoding — see the modeling-approach markdown above).

All predict λ directly (mean of the Poisson) — no output transformation needed.

In [14]:
# Create XGBRegressor
xgb_model = xgb.XGBRegressor(objective='count:poisson', random_state=42)
xgb_model.fit(X_train, y_train)
xgb_rps = evaluate_booster(xgb_model, eval_df)
print(f'XGB RPS: {xgb_rps:.4f}')

# Create LGBMRegressor
lgb_model = lgb.LGBMRegressor(objective='poisson', random_state=42, verbose=0)
lgb_model.fit(X_train, y_train)
lgb_rps = evaluate_booster(lgb_model, eval_df)
print(f'LGB RPS: {lgb_rps:.4f}')

# Create CatBoostRegressor
cb_model = cb.CatBoostRegressor(loss_function='Poisson', random_state=42, verbose=0)
cb_model.fit(X_train, y_train)
cb_rps = evaluate_booster(cb_model, eval_df)
print(f'CB RPS: {cb_rps:.4f}')

# Create HistGradientBoostingRegressor
hgb_model = HistGradientBoostingRegressor(loss='poisson', random_state=42)
hgb_model.fit(X_train, y_train)
hgb_rps = evaluate_booster(hgb_model, eval_df)
print(f'HGB RPS: {hgb_rps:.4f}')

# Create RandomForestRegressor
rf_model = RandomForestRegressor(criterion='poisson', random_state=42)
rf_model.fit(X_train, y_train)
rf_rps = evaluate_booster(rf_model, eval_df)
print(f'RF RPS: {rf_rps:.4f}')

XGB RPS: 0.1690
LGB RPS: 0.1689
CB RPS: 0.1683
HGB RPS: 0.1697
RF RPS: 0.1730


## PoissonRegressor (GLM) — added for decorrelation

Sixth base learner: `PoissonRegressor`, the classical GLM. Structurally the odd one out — no tree splits, just a weighted sum in log-space (`log(λ) = Xβ`, exactly the parametric form Dixon-Coles assumes). That's the whole point.

- **Standalone RPS will be worse than the trees.** Linear log-model can't learn the nonlinear stuff trees do for free (Elo × form interactions, threshold effects on rest days, etc.). Expected — that's not why it's here.
- **Its value is in disagreement.** Its failure modes are different from the trees': it misses interactions where they overfit noise. When one is wrong, the other often isn't. The stacking meta will exploit that.

**Preprocessing is required** (unlike trees):

- **Imputation** — `PoissonRegressor` cannot accept NaN. Median-impute `form_*`, `rest_days`, `h2h_goal_diff`.
- **Scaling** — features live on wildly different scales (Elo ~1500, form ~1.5, rest_days ~30). Without `StandardScaler` the L2 penalty regularizes coefficients asymmetrically — Elo would get shrunk to zero while form barely feels the penalty.
- **One-hot on `tournament_tier`** — it's 4 ordinal levels, but "tier 3 = 3× tier 1" is almost certainly wrong. One-hot lets each tier get its own coefficient, avoiding the forced linearity.
- **`neutral` and `is_home`** — already 0/1, pass through untouched.

Wrapped in a `Pipeline` so `evaluate_booster` works unchanged: `.predict()` applies the preprocessing internally.

In [15]:
# Add PoissonRegressor as well 
numeric_features = [
    'attacker_elo', 'defender_elo',
    'attacker_form_scored', 'attacker_form_conceded',
    'defender_form_scored', 'defender_form_conceded',
    'attacker_rest_days', 'defender_rest_days',
    'h2h_matches_played', 'h2h_goal_diff',
]

numeric_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
])

preprocessor = ColumnTransformer(
    [
        ('num', numeric_pipeline, numeric_features),
        ('tier', OneHotEncoder(handle_unknown='ignore'), ['tournament_tier']),
        # neutral and is_home are passed through (already 0/1)
    ], remainder='passthrough')

glm_model = Pipeline([
    ('prep', preprocessor),
    ('glm', PoissonRegressor(alpha=1.0, max_iter=1000))
])

glm_model.fit(X_train, y_train)
glm_rps = evaluate_booster(glm_model, eval_df) # Same evaluate_booster as before (Never the name)
print(f'GLM RPS: {glm_rps:.4f}')

GLM RPS: 0.1789


## Diversity check — pairwise λ correlations

Before stacking, look at how independent the base learners' predictions actually are. Stacking only pays off when learners disagree in useful ways — if every model produces near-identical λ's, the meta collapses to an average and there's nothing to combine.

Rough guide for interpreting the matrix:

- **>0.99** — redundant. Prune the worse learner of the pair.
- **0.95–0.98** — the sweet spot. Enough disagreement for the meta to weight intelligently.
- **<0.90** — genuine decorrelation. Even a worse standalone RPS is worth keeping — the meta uses it to correct the others' errors in specific regions.

Expected shape: the four boosters cluster tight (same gradient-Poisson recipe on the same features), RF sits a bit lower (bagged + random feature subsets = different variance profile), and the GLM sits lowest (no splits at all, pure log-linear structure). If that pattern holds, the stack is well-designed.

In [16]:
# View correlations between models
lambdas = pd.DataFrame({
    'xgb': xgb_model.predict(reshape_to_long_training(eval_df)[features]),
    'lgb': lgb_model.predict(reshape_to_long_training(eval_df)[features]),
    'cb':  cb_model.predict(reshape_to_long_training(eval_df)[features]),
    'hgb': hgb_model.predict(reshape_to_long_training(eval_df)[features]),
    'rf':  rf_model.predict(reshape_to_long_training(eval_df)[features]),
    'glm': glm_model.predict(reshape_to_long_training(eval_df)[features])
})

print(lambdas.corr())

          xgb       lgb        cb       hgb        rf       glm
xgb  1.000000  0.973724  0.973021  0.972650  0.931326  0.892634
lgb  0.973724  1.000000  0.981305  0.989335  0.939842  0.910568
cb   0.973021  0.981305  1.000000  0.978874  0.934301  0.913253
hgb  0.972650  0.989335  0.978874  1.000000  0.936172  0.903094
rf   0.931326  0.939842  0.934301  0.936172  1.000000  0.868512
glm  0.892634  0.910568  0.913253  0.903094  0.868512  1.000000


## Tuning — Optuna (Bayesian search)

Each learner tuned in its own cell with Optuna's TPE sampler — samples smart candidates based on past trials, converges in ~50–100 trials where random search needs ~300+. `direction='minimize'` on eval RPS, seeded sampler for reproducibility.

**Recency weighting `ξ`** is treated as one more hyperparameter (search range 0.0 → 0.5) — tuned *jointly* with the tree/regularization knobs, passed as `sample_weight = exp(-ξ * age_years)`.

## Inner validation split — for early stopping

Boosters use early stopping (fit up to a high ceiling, quit when a validation score stops improving) instead of tuning `n_estimators` directly. But early stopping needs to *watch* a set — and it can't be `eval_df`, or the model gets tuned against the very set we grade it on.

Solution: carve **inner_val** from the tail of `long_train` (last 6 months, from `2023-07-01` on). Three roles:

- **inner_train** — trees fit on this
- **inner_val** — early stopping watches this
- **eval_df** — Optuna's objective scores this (untouched by training)

After each study, refit on the full `long_train` (inner_train + inner_val) using the winning config.

In [17]:
# Break the long_train into inner_train and inner_val; For early_stopping_rounds validation set 
val_cutoff = pd.Timestamp('2023-07-01') # Last 6 months of train
inner_train_mask = long_train['date'] < val_cutoff
inner_val_mask = long_train['date'] >= val_cutoff

X_inner_train = long_train.loc[inner_train_mask, features]
y_inner_train = long_train.loc[inner_train_mask, 'goals']
X_inner_val = long_train.loc[inner_val_mask, features]
y_inner_val = long_train.loc[inner_val_mask, 'goals']

## Row ages for recency weighting

Precompute `age_years` for every training row once — every Optuna trial reuses these arrays. Anchor is `2024-01-01` (the train/eval boundary), so ages are always positive. Two versions: one over `inner_train` (for tuning trials), one over the full `long_train` (for the final refit).

In [18]:
# Precompute row ages in years for recency weighting 
# ξ joint across all learners: eacg trial rebuilds sample_weight = exp(-ξ * age_years)
reference_date = pd.Timestamp('2024-01-01') # eval split boundary
age_years_inner_train = (reference_date - long_train.loc[inner_train_mask, 'date']).dt.days.values / 365.25
age_years_full_train = (reference_date - long_train['date']).dt.days.values / 365.25

## CatBoost

In [19]:
# Tune CatBoost 
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_cb(trial):
    # 1. Ask Optuna to try this round
    params = {
        'learning_rate':    trial.suggest_float('learning_rate', 0.001, 0.1, log=True),
        'depth':            trial.suggest_int('depth', 6, 10),
        'l2_leaf_reg':      trial.suggest_float('l2_leaf_reg', 2.0, 6.0, log=True),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 20, 80),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'rsm':              trial.suggest_float('rsm', 0.3, 0.9),
    }

    # Add xi (weight decay) as well
    xi = trial.suggest_float('xi', 0.01, 0.1)
    weights_inner_train = np.exp(-xi * age_years_inner_train)

    # 2. Build the model. High iteration ceiling + early stopping 
    model = cb.CatBoostRegressor(
        loss_function='Poisson',
        iterations=1500,
        early_stopping_rounds=50,
        bootstrap_type='Bernoulli',
        random_state=42,
        verbose=0,
        **params
    )

    # 3. Fit on inner_train, watch inner_val for early stopping
    model.fit(
        X_inner_train, y_inner_train,
        sample_weight=weights_inner_train,
        eval_set=(X_inner_val, y_inner_val)
    )

    # 4. Save where early stopping landed - needed for the refit later
    trial.set_user_attr('best_iteration', model.tree_count_)

    # 5. Return the score Optuna should minimize (eval RPS)
    return evaluate_booster(model, eval_df)

# Run the study
study_cb = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study_cb.optimize(objective_cb, n_trials=20, show_progress_bar=True)

print(f'Best CatBoost eval RPS: {study_cb.best_value:.4f}')
print(f'Best params: {study_cb.best_params}')
print(f'Best iteration count: {study_cb.best_trial.user_attrs["best_iteration"]}')

  0%|          | 0/20 [00:00<?, ?it/s]

Best CatBoost eval RPS: 0.1680
Best params: {'learning_rate': 0.01601900833193404, 'depth': 8, 'l2_leaf_reg': 5.595133938484383, 'min_data_in_leaf': 37, 'subsample': 0.8704751081520137, 'rsm': 0.7392607539000343, 'xi': 0.07407800219920707}
Best iteration count: 604


## Refit on full `long_train` with the winning config

Hyperparams and `xi` frozen at the study's best values; `iterations` frozen at the trial's `best_iteration` (no more early stopping). Fits on the full training data — `inner_val` was only there to find the config.

In [ ]:
best_params = study_cb.best_params.copy()
best_xi = best_params.pop('xi')
best_iters = study_cb.best_trial.user_attrs['best_iteration']

# Sample weights on FULL long_train this time
weights_full_train = np.exp(-best_xi * age_years_full_train)

cb_tuned = cb.CatBoostRegressor(
    loss_function='Poisson',
    iterations=best_iters,
    bootstrap_type='Bernoulli',
    random_state=42,
    verbose=0,
    **best_params,
)

cb_tuned.fit(X_train, y_train, sample_weight=weights_full_train)
cb_tuned_rps = evaluate_booster(cb_tuned, eval_df)
print(f'CatBoost tuned (full refit) RPS: {cb_tuned_rps:.4f}')

## LightGBM

In [21]:
def objective_lgb(trial):

    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 125),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 80),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0001, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0001, 10.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'subsample_freq': 1,
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 0.9)
    }

    xi = trial.suggest_float('xi', 0.01, 0.1)
    weights_inner_train = np.exp(-xi * age_years_inner_train)

    model = lgb.LGBMRegressor(
        objective='poisson',
        n_estimators=1500,
        random_state=42,
        verbose=-1,
        **params
    )

    model.fit(
        X_inner_train, y_inner_train,
        sample_weight=weights_inner_train,
        eval_set=[(X_inner_val, y_inner_val)],
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )

    trial.set_user_attr('best_iteration', model.best_iteration_)
    return evaluate_booster(model, eval_df)

study_lgb = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study_lgb.optimize(objective_lgb, n_trials=20, show_progress_bar=True)

print(f'Best LightGBM eval RPS: {study_lgb.best_value:.4f}')
print(f'Best params: {study_lgb.best_params}')
print(f'Best iteration count: {study_lgb.best_trial.user_attrs["best_iteration"]}')

  0%|          | 0/20 [00:00<?, ?it/s]

Best LightGBM eval RPS: 0.1678
Best params: {'learning_rate': 0.005611516415334507, 'num_leaves': 120, 'max_depth': 10, 'min_child_samples': 56, 'reg_alpha': 0.0006026889128682511, 'reg_lambda': 0.000602521573620386, 'subsample': 0.6232334448672797, 'colsample_bytree': 0.8197056874649611, 'xi': 0.0641003510568888}
Best iteration count: 1235


In [ ]:
best_params = study_lgb.best_params.copy()
best_xi = best_params.pop('xi')
best_iters = study_lgb.best_trial.user_attrs['best_iteration']
best_params['subsample_freq'] = 1   # not in Optuna search space (hardcoded in the objective), add back for refit

weights_full_train = np.exp(-best_xi * age_years_full_train)

lgb_tuned = lgb.LGBMRegressor(
    objective='poisson',
    n_estimators=best_iters,
    random_state=42,
    verbose=-1,
    **best_params,
)

lgb_tuned.fit(X_train, y_train, sample_weight=weights_full_train)
lgb_tuned_rps = evaluate_booster(lgb_tuned, eval_df)
print(f'LGB tuned (full refit) RPS: {lgb_tuned_rps:.4f}')

## XGBoost

In [23]:
def objective_xgb(trial):

    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0001, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0001, 10.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 0.9),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0)
    }

    xi = trial.suggest_float('xi', 0.01, 0.1)
    weights_inner_train = np.exp(-xi * age_years_inner_train)

    model = xgb.XGBRegressor(
        objective='count:poisson',
        n_estimators=1500,
        early_stopping_rounds=50,
        random_state=42,
        verbosity=0,
        **params
    )

    model.fit(
        X_inner_train, y_inner_train,
        sample_weight = weights_inner_train,
        eval_set=[(X_inner_val, y_inner_val)],
        verbose=False
    )

    trial.set_user_attr('best_iteration', model.best_iteration)
    return evaluate_booster(model, eval_df)

study_xgb = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study_xgb.optimize(objective_xgb, n_trials=20, show_progress_bar=True)

print(f'Best XGBoost eval RPS: {study_xgb.best_value:.4f}')
print(f'Best params: {study_xgb.best_params}')
print(f'Best iteration count: {study_xgb.best_trial.user_attrs["best_iteration"]}')

  0%|          | 0/20 [00:00<?, ?it/s]

Best XGBoost eval RPS: 0.1678
Best params: {'learning_rate': 0.048376335012497086, 'max_depth': 10, 'min_child_weight': 4, 'reg_alpha': 0.008911826121168483, 'reg_lambda': 1.4254916545552998, 'subsample': 0.8121430808458112, 'colsample_bytree': 0.8129934801649574, 'gamma': 1.664120814330841, 'xi': 0.039344238799575416}
Best iteration count: 163


In [24]:
best_params = study_xgb.best_params.copy()

best_xi = best_params.pop('xi')
weights_full_train = np.exp(-best_xi * age_years_full_train)

xgb_tuned = xgb.XGBRegressor(
    objective='count:poisson',
    n_estimators=1500,
    early_stopping_rounds=50,
    random_state=42,
    verbosity=0,
    **best_params,
)

xgb_tuned.fit(
    X_inner_train, y_inner_train,
    sample_weight=np.exp(-best_xi * age_years_inner_train),
    eval_set=[(X_inner_val, y_inner_val)],
    verbose=False,
)

xgb_tuned_rps = evaluate_booster(xgb_tuned, eval_df)   
print(f'XGB tuned (full refit) RPS: {xgb_tuned_rps:.4f}')

XGB tuned (full refit) RPS: 0.1678


## HistGB

In [25]:
def objective_hgb(trial):

    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.1, log=True),
        'max_iter': trial.suggest_int('max_iter', 100, 1500),
        'max_leaf_nodes': trial.suggest_int('max_leaf_nodes', 15, 127),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 20, 80),
        'l2_regularization': trial.suggest_float('l2_regularization', 0.0001, 10.0, log=True)
    }
    
    xi = trial.suggest_float('xi', 0.01, 0.1)
    weights_inner_train = np.exp(-xi * age_years_inner_train)

    model = HistGradientBoostingRegressor(
        loss='poisson',
        early_stopping=False, # tuning max_iter directly instead
        random_state=42,
        **params
    )

    model.fit(X_inner_train, y_inner_train, sample_weight=weights_inner_train)
    trial.set_user_attr('best_iteration', model.n_iter_) # will equal max_iter since no early stopping

    return evaluate_booster(model, eval_df)

study_hgb = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study_hgb.optimize(objective_hgb, n_trials=20, show_progress_bar=True)

print(f'Best HistGB eval RPS: {study_hgb.best_value:.4f}')
print(f'Best params: {study_hgb.best_params}')

  0%|          | 0/20 [00:00<?, ?it/s]

Best HistGB eval RPS: 0.1681
Best params: {'learning_rate': 0.020046179107204475, 'max_iter': 821, 'max_leaf_nodes': 36, 'max_depth': 11, 'min_samples_leaf': 57, 'l2_regularization': 0.009198958410268715, 'xi': 0.07327453036796801}


In [29]:
best_params = study_hgb.best_params.copy()
best_xi = best_params.pop('xi')
weights_inner_train = np.exp(-best_xi * age_years_inner_train)

hgb_tuned = HistGradientBoostingRegressor(
    loss='poisson',
    early_stopping=False,
    random_state=42,
    **best_params,
)
hgb_tuned.fit(X_inner_train, y_inner_train, sample_weight=weights_inner_train)

hgb_tuned_rps = evaluate_booster(hgb_tuned, eval_df)
print(f'HistGB tuned RPS: {hgb_tuned_rps:.4f}')   # should be 0.1681

HistGB tuned RPS: 0.1681


## RandomForest

In [27]:
def objective_rf(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 200, 800),
        'max_depth':         trial.suggest_int('max_depth', 8, 30),
        'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 5, 50),
        'min_samples_split': trial.suggest_int('min_samples_split', 5, 50),
        'max_features':      trial.suggest_float('max_features', 0.3, 0.9),
    }

    xi = trial.suggest_float('xi', 0.01, 0.1)
    weights_inner_train = np.exp(-xi * age_years_inner_train)

    model = RandomForestRegressor(
        criterion='poisson',
        random_state=42,
        n_jobs=-1,       
        **params,
    )

    model.fit(X_inner_train, y_inner_train, sample_weight=weights_inner_train)
    return evaluate_booster(model, eval_df)


study_rf = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study_rf.optimize(objective_rf, n_trials=20, show_progress_bar=True)

print(f'Best RF eval RPS: {study_rf.best_value:.4f}')
print(f'Best params: {study_rf.best_params}')

  0%|          | 0/20 [00:00<?, ?it/s]

Best RF eval RPS: 0.1681
Best params: {'n_estimators': 565, 'max_depth': 11, 'min_samples_leaf': 7, 'min_samples_split': 48, 'max_features': 0.8793792198447357, 'xi': 0.0827557613304815}


In [28]:
best_params = study_rf.best_params.copy()
best_xi = best_params.pop('xi')
weights_full_train = np.exp(-best_xi * age_years_full_train)

rf_tuned = RandomForestRegressor(
    criterion='poisson',
    random_state=42,
    n_jobs=-1,
    **best_params,
)

rf_tuned.fit(X_train, y_train, sample_weight=weights_full_train)

rf_tuned_rps = evaluate_booster(rf_tuned, eval_df)
print(f'RandomForest tuned (full refit) RPS: {rf_tuned_rps:.4f}')

RandomForest tuned (full refit) RPS: 0.1680


## PoissonRegressor

In [30]:
def objective_glm(trial):
    
    alpha = trial.suggest_float('alpha', 0.001, 10.0, log=True)
    xi = trial.suggest_float('xi', 0.01, 0.1)
    weights_inner_train = np.exp(-xi * age_years_inner_train)

    numeric_pipeline = Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler())
    ])

    preprocessor = ColumnTransformer([
        ('num', numeric_pipeline, numeric_features),
        ('tier', OneHotEncoder(handle_unknown='ignore'), ['tournament_tier']),
    ], remainder='passthrough')

    model = Pipeline([
        ('prep', preprocessor),
        ('glm', PoissonRegressor(alpha=alpha, max_iter=1000))
    ])

    # Pipeline forwards sample_weight to the final step via the "glm__sample_weight" kwarg
    model.fit(
        X_inner_train, y_inner_train,
        glm__sample_weight = weights_inner_train
    )

    return evaluate_booster(model, eval_df)

study_glm = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study_glm.optimize(objective_glm, n_trials=20, show_progress_bar=True)

print(f'Best GLM eval RPS: {study_glm.best_value:.4f}')
print(f'Best params: {study_glm.best_params}')

  0%|          | 0/20 [00:00<?, ?it/s]

Best GLM eval RPS: 0.1688
Best params: {'alpha': 0.03148911647956861, 'xi': 0.09556428757689246}


In [31]:
best_params = study_glm.best_params.copy()
best_xi = best_params.pop('xi')
best_alpha = best_params.pop('alpha')

weights_full_train = np.exp(-best_xi * age_years_full_train)

numeric_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('tier', OneHotEncoder(handle_unknown='ignore'), ['tournament_tier']),
], remainder='passthrough')

glm_tuned = Pipeline([
    ('prep', preprocessor),
    ('glm', PoissonRegressor(alpha=best_alpha, max_iter=1000)),
])

glm_tuned.fit(X_train, y_train, glm__sample_weight=weights_full_train)

glm_tuned_rps = evaluate_booster(glm_tuned, eval_df)
print(f'GLM tuned (full refit) RPS: {glm_tuned_rps:.4f}')

GLM tuned (full refit) RPS: 0.1689


## Plain-average baseline

Before building a fitted stacking meta, measure what a **plain arithmetic mean** of the 6 tuned λ's scores. On small data this is a brutally tough baseline — combining decorrelated learners already extracts most of the value, and a fitted meta with 6 coefficients often just chases noise around the average.

If the meta later can't clearly beat this number, use the average instead. Simpler, more robust, one less layer of overfitting risk.

*Caveat:* the individual learners were tuned to minimize eval RPS, so their eval predictions (and therefore this average) carry the same optimistic bias. Fine for comparing against the fitted meta (equally biased), not a final headline number — that comes from WC2026.

In [34]:
xgb_lambda = xgb_tuned.predict(reshape_to_long_training(eval_df)[features])
lgb_lambda = lgb_tuned.predict(reshape_to_long_training(eval_df)[features])
cb_lambda  = cb_tuned.predict(reshape_to_long_training(eval_df)[features])
hgb_lambda = hgb_tuned.predict(reshape_to_long_training(eval_df)[features])
rf_lambda  = rf_tuned.predict(reshape_to_long_training(eval_df)[features])
glm_lambda = glm_tuned.predict(reshape_to_long_training(eval_df)[features])

combined_lambda = (xgb_lambda + lgb_lambda + cb_lambda + hgb_lambda + rf_lambda + glm_lambda) / 6

n = len(eval_df)
lambda_home = combined_lambda[:n]
lambda_away = combined_lambda[n:]

home_score = eval_df['home_score'].to_numpy()
away_score = eval_df['away_score'].to_numpy()

rps_values = []
for i in range(n):
    home_probs = poisson.pmf(np.arange(11), lambda_home[i])
    away_probs = poisson.pmf(np.arange(11), lambda_away[i])
    matrix = np.outer(home_probs, away_probs)
    probs = [
        np.tril(matrix, -1).sum(), 
        np.trace(matrix), 
        np.triu(matrix, 1).sum()
    ]

    if home_score[i] > away_score[i]:
        actual = [1, 0, 0]
    elif home_score[i] == away_score[i]:
        actual = [0, 1, 0]
    else:
        actual = [0, 0, 1]

    rps = ranked_probability_score(probs, actual)
    rps_values.append(rps)

print(f'Plain-average baseline RPS: {np.mean(rps_values):.4f}')

Plain-average baseline RPS: 0.1674


## OOF fit functions

One helper per learner. Each reads the tuned config directly from its `study_*` object (no rounding), pops `xi`, and returns a fresh fitted model.

In [37]:
def fit_cb(X, y, w):
    params = study_cb.best_params.copy()
    params.pop('xi')
    n_iters = study_cb.best_trial.user_attrs['best_iteration']

    m = cb.CatBoostRegressor(
        loss_function='Poisson', iterations=n_iters,
        bootstrap_type='Bernoulli', random_state=42, verbose=0, **params,
    )

    m.fit(X, y, sample_weight=w)
    return m

def fit_lgb(X, y, w):
    params = study_lgb.best_params.copy()
    params.pop('xi')
    params['subsample_freq'] = 1                     # not in Optuna search, add back
    n_iters = study_lgb.best_trial.user_attrs['best_iteration']

    m = lgb.LGBMRegressor(
        objective='poisson', n_estimators=n_iters,
        random_state=42, verbose=-1, **params,
    )

    m.fit(X, y, sample_weight=w)
    return m

def fit_xgb(X, y, w):
    params = study_xgb.best_params.copy()
    params.pop('xi')
    n_iters = study_xgb.best_trial.user_attrs['best_iteration']

    m = xgb.XGBRegressor(
        objective='count:poisson', n_estimators=n_iters,
        random_state=42, verbosity=0, **params,
    )

    m.fit(X, y, sample_weight=w)
    return m

def fit_hgb(X, y, w):
    params = study_hgb.best_params.copy()
    params.pop('xi')

    m = HistGradientBoostingRegressor(
        loss='poisson', early_stopping=False, random_state=42, **params,
    )

    m.fit(X, y, sample_weight=w)
    return m

def fit_rf(X, y, w):
    params = study_rf.best_params.copy()
    params.pop('xi')

    m = RandomForestRegressor(
        criterion='poisson', random_state=42, n_jobs=-1, **params,
    )

    m.fit(X, y, sample_weight=w)
    return m

def fit_glm(X, y, w):
    params = study_glm.best_params.copy()
    params.pop('xi')
    alpha = params.pop('alpha')

    prep = ColumnTransformer([
        ('num', Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale', StandardScaler()),
        ]), numeric_features),
        ('tier', OneHotEncoder(handle_unknown='ignore'), ['tournament_tier']),
    ], remainder='passthrough')

    m = Pipeline([
        ('prep', prep), 
        ('glm', PoissonRegressor(alpha=alpha, max_iter=1000))
    ])

    m.fit(X, y, glm__sample_weight=w)
    return m


# Per-learner ξ (each learner picked its own)
xi_dict = {
    'cb':  study_cb.best_params['xi'],
    'lgb': study_lgb.best_params['xi'],
    'xgb': study_xgb.best_params['xi'],
    'hgb': study_hgb.best_params['xi'],
    'rf':  study_rf.best_params['xi'],
    'glm': study_glm.best_params['xi'],
}

fit_funcs = {
    'cb':  fit_cb,
    'lgb': fit_lgb,
    'xgb': fit_xgb,
    'hgb': fit_hgb,
    'rf':  fit_rf,
    'glm': fit_glm,
}

## OOF fold loop — 3 expanding-window folds

For each fold: fit the 6 tuned learners on data before the boundary, predict λ on the OOF window. Every training row ends up with 6 λ predictions from models that never saw it — the meta trains on these.

In [44]:
# Each tuple = (fit_end, oof_end) — fit on rows before fit_end, predict on rows in [fit_end, oof_end).
# Expanding window: fold 2's fit_slice includes fold 1's oof rows too, and so on.
fold_boundaries = [
    (pd.Timestamp('2008-01-01'), pd.Timestamp('2013-01-01')),
    (pd.Timestamp('2013-01-01'), pd.Timestamp('2018-01-01')),
    (pd.Timestamp('2018-01-01'), pd.Timestamp('2024-01-01')),  # oof_end = train_df boundary
]

oof_chunks = []  # one DataFrame per fold, concatenated at the end

for fold_idx, (fit_end, oof_end) in enumerate(fold_boundaries, 1):
    print(f'\n--- Fold {fold_idx}: fit on <{fit_end.date()}, OOF on [{fit_end.date()}, {oof_end.date()}) ---')

    # 1. Split train_df in time. fit_slice = training history, oof_slice = held-out chunk for this fold.
    fit_slice = train_df[train_df['date'] < fit_end]
    oof_slice = train_df[(train_df['date'] >= fit_end) & (train_df['date'] < oof_end)]

    # 2. Reshape both to attacker/defender long format (2 rows per match) — same layout base learners expect.
    long_fit = reshape_to_long_training(fit_slice)
    long_oof = reshape_to_long_training(oof_slice)

    X_fit = long_fit[features]     # features for training
    y_fit = long_fit['goals']      # target (goals scored per attacking side)
    X_oof = long_oof[features]     # features to predict on

    # 3. Row ages for recency weights — recomputed per fold since fit_slice grows each iteration.
    age_years_fit = (reference_date - long_fit['date']).dt.days.values / 365.25

    n_oof = len(oof_slice)  # number of MATCHES (not long rows) in the OOF window

    # 4. Container for this fold's output — one row per match, columns filled in below.
    fold_preds = pd.DataFrame({
        'match_id':   oof_slice['match_id'].values,
        'date':       oof_slice['date'].values,
        'home_score': oof_slice['home_score'].values,   # true target the meta will learn
        'away_score': oof_slice['away_score'].values,
    })

    # 5. Fit each tuned learner, predict on the OOF slice, save its λ_home and λ_away columns.
    for name, fit_func in fit_funcs.items():
        print(f'  fitting {name}...')
        # Each learner has its own tuned ξ → its own recency weights on the same fit_slice.
        w = np.exp(-xi_dict[name] * age_years_fit)

        model = fit_func(X_fit, y_fit, w)   # fresh fit, tuned config from study_* (no rounding)
        preds = model.predict(X_oof)        # length 2 * n_oof: first half = home-side λ, second = away-side λ

        fold_preds[f'{name}_lambda_home'] = preds[:n_oof]
        fold_preds[f'{name}_lambda_away'] = preds[n_oof:]

    oof_chunks.append(fold_preds)

# Concat all folds → the meta's training set. Every row has 6 λ predictions from models that DIDN'T see it.
oof_df = pd.concat(oof_chunks, ignore_index=True)
print(f'\nOOF table: {len(oof_df)} matches × {len(oof_df.columns)} cols')
print(oof_df.head())


--- Fold 1: fit on <2008-01-01, OOF on [2008-01-01, 2013-01-01) ---
  fitting cb...
  fitting lgb...
  fitting xgb...
  fitting hgb...
  fitting rf...
  fitting glm...

--- Fold 2: fit on <2013-01-01, OOF on [2013-01-01, 2018-01-01) ---
  fitting cb...
  fitting lgb...
  fitting xgb...
  fitting hgb...
  fitting rf...
  fitting glm...

--- Fold 3: fit on <2018-01-01, OOF on [2018-01-01, 2024-01-01) ---
  fitting cb...
  fitting lgb...
  fitting xgb...
  fitting hgb...
  fitting rf...
  fitting glm...

OOF table: 15299 matches × 16 cols
   match_id       date  home_score  away_score  cb_lambda_home  \
0     23601 2008-01-02           3           2        1.796130   
1     23602 2008-01-05           3           0        2.800863   
2     23603 2008-01-06           1           2        1.663192   
3     23604 2008-01-08           1           0        1.595795   
4     23605 2008-01-09           2           0        1.580588   

   cb_lambda_away  lgb_lambda_home  lgb_lambda_away  xgb_lam

## Stacking meta — penalized PoissonRegressor

Reshape `oof_df` into home + away rows (same attacker/defender trick as base learners), so each row is one Poisson target with 6 λ features. Fit a `PoissonRegressor` with L2 penalty. `alpha=1.0` is a first pass — will tune next.

In [45]:
lambda_learners = ['cb', 'lgb', 'xgb', 'hgb', 'rf', 'glm']

# Home side rows
home_rows = oof_df.rename(columns={f'{n}_lambda_home': f'{n}_lambda' for n in lambda_learners})
home_rows = home_rows[[f'{n}_lambda' for n in lambda_learners] + ['home_score']].rename(columns={'home_score': 'goals'})

# Away side rows
away_rows = oof_df.rename(columns={f'{n}_lambda_away': f'{n}_lambda' for n in lambda_learners})
away_rows = away_rows[[f'{n}_lambda' for n in lambda_learners] + ['away_score']].rename(columns={'away_score': 'goals'})

meta_train = pd.concat([home_rows, away_rows], ignore_index=True)

meta_features = [f'{n}_lambda' for n in lambda_learners]
X_meta = meta_train[meta_features]
y_meta = meta_train['goals']

# Fix the meta (L2 only, alpha=1.0 as first pass ~ tune next)
meta = PoissonRegressor(alpha=1.0, max_iter=1000)
meta.fit(X_meta, y_meta)

print(f'Meta intercept: {meta.intercept_:.4f}')
print('Meta coefficients:')
for name, coef in zip(meta_features, meta.coef_):
    print(f'  {name}: {coef:+.4f}')

Meta intercept: -0.2968
Meta coefficients:
  cb_lambda: +0.0673
  lgb_lambda: +0.0548
  xgb_lambda: +0.0740
  hgb_lambda: +0.0491
  rf_lambda: +0.0762
  glm_lambda: +0.0681
